# Explora aquí

Se recomienda utilizar este cuaderno con fines de exploración.

In [4]:
pip install pandas requests lxml


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [20]:
import os
from bs4 import BeautifulSoup
import requests
import time
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO
import io



url = "https://en.wikipedia.org/wiki/List_of_most-streamed_songs_on_Spotify"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

if response.status_code != 200:
    raise Exception(f"Error accessing the page: {response.status_code}")

print("Status:", response.status_code)

Status: 200


In [21]:
html = io.StringIO(response.text)
tables = pd.read_html(html)
print(f"Se encontraron {len(tables)} tablas.")

Se encontraron 30 tablas.


In [23]:
df = tables[0]
df.head()

,Rank,Song,Artist(s),Streams (billions),Release date,Ref.
0,1,"""Blinding Lights""",The Weeknd,5.379,29 November 2019,[1]
1,2,"""Shape of You""",Ed Sheeran,4.875,6 January 2017,[2]
2,3,"""Sweater Weather""",The Neighbourhood,4.548,3 December 2012,[3]
3,4,"""Starboy""",The Weeknd and Daft Punk,4.492,21 September 2016,[4]
4,5,"""As It Was""",Harry Styles,4.371,1 April 2022,[5]


In [26]:
df["Streams (billions)"] = df["Streams (billions)"].astype(str)
df["Streams (billions)"] = df["Streams (billions)"].str.replace(r"[^\d.]", "", regex=True)
df["Streams (billions))"] = df["Streams (billions)"].astype(float)
 
df.head()

,Rank,Song,Artist(s),Streams (billions),Release date,Ref.,Streams (billions))
0,1,"""Blinding Lights""",The Weeknd,5.379,29 November 2019,[1],5.379
1,2,"""Shape of You""",Ed Sheeran,4.875,6 January 2017,[2],4.875
2,3,"""Sweater Weather""",The Neighbourhood,4.548,3 December 2012,[3],4.548
3,4,"""Starboy""",The Weeknd and Daft Punk,4.492,21 September 2016,[4],4.492
4,5,"""As It Was""",Harry Styles,4.371,1 April 2022,[5],4.371


In [27]:
df.sort_values(by="Streams (billions)", ascending=False).head()

,Rank,Song,Artist(s),Streams (billions),Release date,Ref.,Streams (billions))
0,1,"""Blinding Lights""",The Weeknd,5.379,29 November 2019,[1],5.379
1,2,"""Shape of You""",Ed Sheeran,4.875,6 January 2017,[2],4.875
2,3,"""Sweater Weather""",The Neighbourhood,4.548,3 December 2012,[3],4.548
3,4,"""Starboy""",The Weeknd and Daft Punk,4.492,21 September 2016,[4],4.492
4,5,"""As It Was""",Harry Styles,4.371,1 April 2022,[5],4.371


In [35]:
conn = sqlite3.connect("spotify_top_songs.db")

df.to_sql("most_streamed", conn, if_exists="replace", index=False)

conn.commit()

conn.close()
